In [1]:
import sys
import argparse
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import numpy as np
import anndata
import tqdm
import pandas as pd






In [38]:

# --- INIT ---

adata_path = '/Users/mingyaolab/Dropbox/aorta_circadian_data/datasets/joint/adata_qc_filtered.h5ad'
cluster_path = '/Users/mingyaolab/Dropbox/aorta_circadian_data/datasets/joint/data_annotations/hvg/prior_knowledge_guided/scvi_res/clustering/res_0.05/subclustering/res_0.2/smc_subclusters.tsv'
reg_head_folder = '/users/mingyaolab/desktop/chunk_results'
if not os.path.exists(reg_head_folder):
    os.makedirs(reg_head_folder)
min_prop = 1e-7
num_genes_per_chunk = 2000



In [5]:
# --- LOAD ADATA ---

adata = anndata.read_h5ad(adata_path)

adata


AnnData object with n_obs × n_vars = 145271 × 32285
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'

In [6]:
# --- ADD EMBEDDINGS TO ADATA ---



# ** load clusters **
cluster_df = pd.read_table(cluster_path,sep='\t',index_col='index')

# ** make sure everything in the same order
adata = adata[list(cluster_df.index)]


# ** add embeddings **
adata.obs["cluster"] = np.array(cluster_df['smc_subcluster'])

adata


/var/folders/y2/0zd17m8j6bz5j5xv0cy2dw680000gp/T/ipykernel_99932/2564887669.py:13: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["cluster"] = np.array(cluster_df['smc_subcluster'])


AnnData object with n_obs × n_vars = 145271 × 32285
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'

In [32]:
# --- GET THE UNIQUE DESCRIPTION AND CLUSTERS ---

descriptions = list(adata.obs['description'].unique())
clusters = sorted(list(adata.obs['cluster'].unique()))
clusters = list(filter(lambda x: ~np.isnan(x),clusters)) # get rid of NaN cluster (only relevant for SMC subcluster)
clusters = list(map(lambda x: int(x),clusters))



print("Unique descriptions:\n",descriptions)
print("Unique clusters:\n",clusters)


# --- LIMIT ADATA TO GENES THAT MEET MINIMUM PSEUDOBULK COUNT THRESHOLD ---


# ** get gene pseudobulk counts
adata.var['pseudobulk_count'] = np.array(np.sum(adata.X,axis=0)).flatten()

# ** get the number of cells in the smallest cell type
smallest_cell_type_adata = adata[adata.obs['cluster'] == np.max(clusters)]

# ** get pseudobulk cutoff **
pseudobulk_threshold = min_prop * np.sum(smallest_cell_type_adata.obs['lib_size'])

# ** limit adata to this **
adata = adata[:,adata.var['pseudobulk_count'] >= pseudobulk_threshold]


adata


Unique descriptions:
 ['male aligned bmal1-ko', 'male misaligned bmal1-control', 'female aligned bmal1-ko', 'female misaligned bmal1-control', 'male aligned bmal1-control', 'female aligned bmal1-control']
Unique clusters:
 [0, 1, 2, 3, 4, 5, 6, 7]


View of AnnData object with n_obs × n_vars = 145271 × 18868
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2', 'pseudobulk_count'

In [9]:
# --- IDENTIFY GENES MEETING THE MINIMUM PROP IN EACH CLUSTER / CONDITION COMBO ---

import warnings
warnings.filterwarnings('ignore')

genes_to_est = set()
for i, description in enumerate(descriptions):
    for j, cluster in enumerate(clusters):
        print("Description: %s; Cluster: %s" % (i,j))
            
        cluster_condition_adata = adata[(adata.obs['cluster'] == cluster) & (adata.obs['description'] == description)]
        cluster_condition_adata.var['prop'] = np.array(np.sum(cluster_condition_adata.X,axis=0)).flatten()  / np.sum(cluster_condition_adata.obs['lib_size'])
        cluster_condition_adata = cluster_condition_adata[:,cluster_condition_adata.var['prop'] >= min_prop]
        genes_to_est.update(list(cluster_condition_adata.var_names))

# make it a list
genes_to_est = list(genes_to_est)



Description: 0; Cluster: 0
Description: 0; Cluster: 1
Description: 0; Cluster: 2
Description: 0; Cluster: 3
Description: 0; Cluster: 4
Description: 0; Cluster: 5
Description: 0; Cluster: 6
Description: 0; Cluster: 7
Description: 1; Cluster: 0
Description: 1; Cluster: 1
Description: 1; Cluster: 2
Description: 1; Cluster: 3
Description: 1; Cluster: 4
Description: 1; Cluster: 5
Description: 1; Cluster: 6
Description: 1; Cluster: 7
Description: 2; Cluster: 0
Description: 2; Cluster: 1
Description: 2; Cluster: 2
Description: 2; Cluster: 3
Description: 2; Cluster: 4
Description: 2; Cluster: 5
Description: 2; Cluster: 6
Description: 2; Cluster: 7
Description: 3; Cluster: 0
Description: 3; Cluster: 1
Description: 3; Cluster: 2
Description: 3; Cluster: 3
Description: 3; Cluster: 4
Description: 3; Cluster: 5
Description: 3; Cluster: 6
Description: 3; Cluster: 7
Description: 4; Cluster: 0
Description: 4; Cluster: 1
Description: 4; Cluster: 2
Description: 4; Cluster: 3
Description: 4; Cluster: 4
D

In [10]:
# --- MAKE THE GENE CHUNKS ---


gene_chunk_indices = list(np.arange(0,len(genes_to_est),num_genes_per_chunk)) + [len(genes_to_est)]
chunk_gene_list = []
for gene_chunk_bin_index in range(0,len(gene_chunk_indices) - 1):
    chunk_start_index = gene_chunk_indices[gene_chunk_bin_index]
    chunk_end_index = gene_chunk_indices[gene_chunk_bin_index + 1]
    chunk_genes = genes_to_est[chunk_start_index:chunk_end_index]
    chunk_gene_list.append(chunk_genes)



In [39]:
# --- MAKE THE FOLDER OUTS ---


# subfolders for each cluster / condition combo
cluster_description_folder_out_dict = {}
for cluster in clusters:
    cluster_reg_folder = '%s/cluster_%s' % (reg_head_folder,cluster)
    if not os.path.exists(cluster_reg_folder):
        os.makedirs(cluster_reg_folder)
    for description in descriptions:
        cluster_description_reg_folder = '%s/%s' % (cluster_reg_folder,description)
        if not os.path.exists(cluster_description_reg_folder):
            os.makedirs(cluster_description_reg_folder)
            
        # update dict
        if cluster not in cluster_description_folder_out_dict:
            cluster_description_folder_out_dict[cluster] = {}
        cluster_description_folder_out_dict[cluster][description] = cluster_description_reg_folder

    
    

In [28]:
# --- WRITE OUT THE GENES TO ESTIMATE ---

genes_to_est_fileout = '%s/genes_to_est.txt' % reg_head_folder
with open(genes_to_est_fileout,"wb") as file_obj:
    file_obj.write("\n".join(genes_to_est).encode())




In [29]:
# --- WRITE OUT THE GENE CHUNKS ---

gene_chunk_folder_out = '%s/gene_chunks' % reg_head_folder
if not os.path.exists(gene_chunk_folder_out):
    os.makedirs(gene_chunk_folder_out)
for i, gene_chunk in enumerate(chunk_gene_list):
    genes_to_est_fileout = '%s/genes_to_est_%s.txt' % (gene_chunk_folder_out,i)
    with open(genes_to_est_fileout,"wb") as file_obj:
        file_obj.write("\n".join(gene_chunk).encode())




In [30]:
# --- GET THE COMMANDS ---


commands = []
for cluster in clusters:
    for description in descriptions:
        for chunk_index, gene_chunk in enumerate(chunk_gene_list):
        
            # get the path out
            path_out = cluster_description_folder_out_dict[cluster][description]
            path_out = "%s/chunk_%s" % (path_out,chunk_index)

            # gene file
            gene_est_path = 'chunk_results/gene_chunks/genes_to_est_%s.txt' % chunk_index
            
            # get command
            command = "python run_non_parametric_reg.py -f adata_qc_filtered.h5ad -gf %s -cf smc_subclusters.tsv -c %s -d '%s' -o '%s'" % (gene_est_path, cluster, description, path_out)

            # add conda activate to command
            command = "source /home/benauer/anaconda2/bin/activate tempo && %s" % command

            # make the job
            job_string = '%s_%s_chunk_%s' % (cluster,description,chunk_index)
            job_string = job_string.replace(" ", "_")
            command = 'bsub -J %s -M 64000 -e error_files/%s.e -o out_files/%s.o -R "rusage[mem=64000]" "%s"' % (job_string,job_string,job_string, command)

            # add
            commands.append(command)
        
        


In [31]:
# --- WRITE THE COMMANDS ---

with open('reg_commands.sh',"wb") as file_obj:
    file_obj.write("\n".join(commands).encode())





In [9]:
# # --- ADD PHASE COL ---

# adata.obs['phase'] = np.array((adata.obs['zt'] / 24.0) * 2 * np.pi)



In [58]:
# # --- CONCAT TOGETHER ---

# for cluster in clusters:
#     for description in descriptions:

#         de_novo_dfs, log_alpha_dfs, log_beta_dfs, min_max_dfs = [], [], [], []
#         for chunk_index, gene_chunk in enumerate(chunk_gene_list):

#             # get the path out
#             path_out = cluster_description_folder_out_dict[cluster][description]
#             path_out = "%s/chunk_%s" % (path_out,chunk_index)

#             # load dfs
#             de_novo_df = pd.read_table('%s/de_novo_metrics.tsv' % path_out,sep='\t',index_col='gene')
#             log_alpha_df = pd.read_table('%s/gene_log_alpha.tsv' % path_out,sep='\t',index_col='gene')
#             log_beta_df = pd.read_table('%s/gene_log_beta.tsv' % path_out,sep='\t',index_col='gene')
#             min_max_df = pd.read_table('%s/log_min_max.tsv' % path_out,sep='\t',index_col='gene')

#             # append to lists
#             de_novo_dfs.append(de_novo_df)
#             log_alpha_dfs.append(log_alpha_df)
#             log_beta_dfs.append(log_beta_df)
#             min_max_dfs.append(min_max_df)


#         # ** concat **
#         de_novo_df = pd.concat(de_novo_dfs)
#         log_alpha_df = pd.concat(log_alpha_dfs)
#         log_beta_df = pd.concat(log_beta_dfs)
#         min_max_df = pd.concat(min_max_dfs)

#         # ** write out **
#         de_novo_df.to_csv('%s/de_novo_metrics.tsv' % cluster_description_folder_out_dict[cluster][description],sep='\t')
#         log_alpha_df.to_csv('%s/gene_log_alpha.tsv' % cluster_description_folder_out_dict[cluster][description],sep='\t')
#         log_beta_df.to_csv('%s/gene_log_beta.tsv' % cluster_description_folder_out_dict[cluster][description],sep='\t')
#         min_max_df.to_csv('%s/log_min_max.tsv' % cluster_description_folder_out_dict[cluster][description],sep='\t')

#         # ** cp the config from chunk 0 **
#         command = 'cp "%s/chunk_0/config.txt" "%s/config.txt"' % (cluster_description_folder_out_dict[cluster][description],cluster_description_folder_out_dict[cluster][description])
#         os.system(command)


